# SEE Detector Full Pipeline

Notebook ini menjalankan pipeline lengkap: train/test split, cross-validation, evaluasi holdout, dan feature importance.

In [1]:
from see_detector_pipeline import load_data, preprocess_dynamic, aggregate_dynamic, build_dataset, train_and_evaluate_full

In [2]:
static, all_dyn = load_data()
print("Static (AMAN+RENTAN):", len(static))
all_dyn.head()

Static (AMAN+RENTAN): 69


,extension_id,extension_name,timestamp,scenario,source,origin,method,url,domain,resource_type,post_data_preview,request_headers_json,is_extension_initiated,is_unauthorized_domain,initiator_url,frame_url,host_permissions,content_script_matches,evidence_summary
0,bbdpihagclfjiodkbebbheamdhifhcgl,"YapThread - Record, Bookmark, AI Chat",2026-08-11T23:12:09.406,S3_Gmail,Service Worker,chrome-extension://bbdpihagclfjiodkbebbheamdhi...,GET,https://api.yapthread.com/trpc/upload.scrapeVi...,api.yapthread.com,fetch,NaN,"{""content-type"": ""application/json"", ""referer""...",True,True,NaN,NaN,"""[]""","""[\""<all_urls>\""]""",GET to api.yapthread.com; from extension servi...
1,bbdpihagclfjiodkbebbheamdhifhcgl,"YapThread - Record, Bookmark, AI Chat",2026-08-11T23:12:09.407,S3_Gmail,Service Worker,chrome-extension://bbdpihagclfjiodkbebbheamdhi...,GET,https://api.yapthread.com/trpc/upload.scrapeVi...,api.yapthread.com,fetch,NaN,"{""content-type"": ""application/json"", ""referer""...",True,True,NaN,NaN,"""[]""","""[\""<all_urls>\""]""",GET to api.yapthread.com; from extension servi...
2,bbdpihagclfjiodkbebbheamdhifhcgl,"YapThread - Record, Bookmark, AI Chat",2026-08-11T23:12:19.855,S3_Gmail,Service Worker,chrome-extension://bbdpihagclfjiodkbebbheamdhi...,GET,https://api.yapthread.com/trpc/upload.scrapeVi...,api.yapthread.com,fetch,NaN,"{""content-type"": ""application/json"", ""referer""...",True,True,NaN,NaN,"""[]""","""[\""<all_urls>\""]""",GET to api.yapthread.com; from extension servi...
3,bbdpihagclfjiodkbebbheamdhifhcgl,"YapThread - Record, Bookmark, AI Chat",2026-08-11T23:12:23.270,S3_Gmail,Service Worker,chrome-extension://bbdpihagclfjiodkbebbheamdhi...,GET,https://api.yapthread.com/trpc/upload.scrapeVi...,api.yapthread.com,fetch,NaN,"{""content-type"": ""application/json"", ""referer""...",True,True,NaN,NaN,"""[]""","""[\""<all_urls>\""]""",GET to api.yapthread.com; from extension servi...
4,bbdpihagclfjiodkbebbheamdhifhcgl,"YapThread - Record, Bookmark, AI Chat",2026-08-11T23:12:43.528,S4_HTTP_Site,Service Worker,chrome-extension://bbdpihagclfjiodkbebbheamdhi...,GET,https://api.yapthread.com/trpc/upload.scrapeVi...,api.yapthread.com,fetch,NaN,"{""content-type"": ""application/json"", ""referer""...",True,True,NaN,NaN,"""[]""","""[\""<all_urls>\""]""",GET to api.yapthread.com; from extension servi...


In [3]:
all_dyn_prep = preprocess_dynamic(all_dyn)
agg_dyn = aggregate_dynamic(all_dyn_prep)
print("Dynamic extensions with traffic:", agg_dyn["extension_id"].nunique())
agg_dyn.head()

Dynamic extensions with traffic: 63


e:\Kuliah\Skripsi\Semhas\extension\see-detector\Perplexity-Pipeline\see_detector_pipeline.py:154: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  agg_dyn = all_dyn.groupby('extension_id').apply(agg_group).reset_index()


,extension_id,dyn_total,dyn_unauth,dyn_extinit,dyn_unauth_ratio,dyn_extinit_ratio,dyn_domains,dyn_scenarios,dyn_sources,dyn_get,...,dyn_post_max_len,dyn_url_exfil_cnt,dyn_has_dom_scraping,dyn_has_injected,dyn_post_json_cnt,dyn_post_structured_cnt,victim_url_count_total,victim_sensitive_total,victim_file_total,victim_hosts_unique_total
0,Backit Plugin__hfdhpmpfpcnbboppkkkblilhbloejij...,3,1,3,0.333333,1.0,2,1,1,3,...,3.0,1,False,False,0,0,0,0,0,0
1,Braavos_ Bitcoin & Starknet Wallet__jnlgamecbp...,8,8,8,1.000000,1.0,4,2,2,1,...,1068.0,4,False,False,7,7,8,2,0,4
2,Brisk Boost__pdldjapecechflnpdgpeiklnbigamggl_...,53,51,53,0.962264,1.0,3,8,2,1,...,668.0,1,False,True,1,19,0,0,0,0
3,Bro-Mon 🐾 Browser Monsters - Web Surf Game__oe...,33,25,33,0.757576,1.0,11,6,3,28,...,3.0,3,False,True,0,0,2,0,0,2
4,Clockify Time Tracker__pmjeegjhjdlccodhacdgbgf...,1,0,1,0.000000,1.0,1,1,1,1,...,3.0,0,False,False,0,0,0,0,0,0


In [4]:
merged, X, y, groups, feature_cols = build_dataset(static, agg_dyn)
print("Dataset size:", len(merged))
print("Label distribution:", merged["y"].value_counts().to_dict())

Dataset size: 69
Label distribution: {1: 39, 0: 30}


In [5]:
model_train, model_all = train_and_evaluate_full(X, y, groups, feature_cols)

Train size: 55 extensions; Test size: 14 extensions
===== Cross-validation on TRAIN (5-fold, GroupKFold) =====
train_accuracy: 1.0000 ± 0.0000
train_precision: 1.0000 ± 0.0000
train_recall: 1.0000 ± 0.0000
train_f1: 1.0000 ± 0.0000
test_accuracy: 0.9636 ± 0.0445
test_precision: 0.9667 ± 0.0667
test_recall: 0.9750 ± 0.0500
test_f1: 0.9685 ± 0.0394
===== Evaluation on TEST HOLDOUT =====
Confusion matrix (TEST):
[[6 2]
 [1 5]]
===== Classification report (TEST):
              precision    recall  f1-score   support

           0     0.8571    0.7500    0.8000         8
           1     0.7143    0.8333    0.7692         6

    accuracy                         0.7857        14
   macro avg     0.7857    0.7917    0.7846        14
weighted avg     0.7959    0.7857    0.7868        14

ROC-AUC (TEST): 0.9375
===== Top 20 feature importances (fit on ALL data) =====
risk_score                     0.3980
http_api_total                 0.1237
external_domains               0.0920
permissions_cou